# Wallat Faithfulness Test — Retrieval Pipeline (Kaggle-adapted)

**Before running:** Settings (right panel) → Internet: **ON** (needed to stream from the HF Hub).
Accelerator can stay **None/CPU** for this whole notebook — BM25 indexing doesn't need a GPU.
Save your T4/P100 quota for the generation stage later.

Run cells top to bottom. Each stage writes its output to `/kaggle/working/`, which
persists across sessions on Kaggle (unlike `/kaggle/temp`), so you can stop and resume
without recomputing earlier stages.

## Stage 0 — Install pinned dependencies

`datasets>=4.0` dropped support for **loading-script** datasets entirely — and `facebook/kilt_wikipedia` / `facebook/kilt_tasks` are still script-based (confirmed via their HF dataset-viewer pages, which show *"requires arbitrary Python code execution"*). So we pin an older `datasets` version that still supports `trust_remote_code=True`. **Restart the kernel after this cell runs once** if Kaggle's preinstalled version conflicts.

In [ ]:
!rm -rf /kaggle/working/*

In [ ]:
!pip install -q "datasets==2.19.0" "fsspec==2023.9.2" rank_bm25
print("Installed. If this is the first install this session, restart the kernel now, then continue from Stage 1.")

## Stage 1 — Config & imports

In [ ]:
import json
import random
import re
from pathlib import Path

from datasets import load_dataset, get_dataset_config_names
from rank_bm25 import BM25Okapi

# ---- Config: tune based on your remaining session time / GPU quota ----
N_QUESTIONS = 500          # verified end-to-end at 100; this is the real run
N_DISTRACTORS = 5000      # random Wikipedia articles added as retrieval "noise"
CHUNK_SIZE_TOKENS = 100    # matches Wallat et al.'s chunking
SEED = 42

# /kaggle/working persists across sessions -- use it, not a relative path
OUT_DIR = Path("/kaggle/working/retrieval_pipeline_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
print("Output directory:", OUT_DIR)

## Stage 1b — Diagnostic: confirm the KILT-NQ config name

Don't assume `"nq"` is correct -- config names for `facebook/kilt_tasks` have shifted before. Run this once to see the real list, then set `NQ_CONFIG` below accordingly.

In [ ]:
try:
    configs = get_dataset_config_names("facebook/kilt_tasks", trust_remote_code=True)
    print("Available configs:", configs)
except Exception as e:
    print("Could not list configs automatically:", e)
    print("Fall back to checking https://huggingface.co/datasets/facebook/kilt_tasks manually.")

In [ ]:
# Set this from the printed list above (commonly "nq" for Natural Questions in KILT).
NQ_CONFIG = "nq"

## Stage 2 — Sample questions from KILT-NQ + record gold provenance

In [ ]:
def sample_questions(n=N_QUESTIONS, config_name=NQ_CONFIG):
    ds = load_dataset("facebook/kilt_tasks", config_name, split="validation", trust_remote_code=True)

    def has_provenance(ex):
        try:
            return len(ex["output"][0]["provenance"]) > 0
        except (IndexError, KeyError, TypeError):
            return False

    ds = ds.filter(has_provenance)
    n = min(n, len(ds))
    indices = random.sample(range(len(ds)), n)
    sampled = ds.select(indices)

    records = []
    gold_ids = set()
    for ex in sampled:
        prov_ids = [p["wikipedia_id"] for p in ex["output"][0]["provenance"]]
        gold_ids.update(prov_ids)
        records.append({
            "id": ex["id"],
            "question": ex["input"],
            "answer": ex["output"][0].get("answer", ""),
            "gold_wikipedia_ids": prov_ids,
        })

    with open(OUT_DIR / "sampled_questions.json", "w") as f:
        json.dump(records, f, indent=2)
    with open(OUT_DIR / "gold_ids.json", "w") as f:
        json.dump(sorted(gold_ids), f)

    print(f"Sampled {len(records)} questions, needing {len(gold_ids)} gold documents.")
    return records, gold_ids

questions, gold_ids = sample_questions()

## Stage 2b - Scan checkpoint (resume if one exists, cold scan if not)

Stage 3 streams all 5.9M KILT Wikipedia docs, which costs ~3.4 hours. This cell makes that
resumable, and runs correctly whether or not a checkpoint is attached:

- **`RESUME_FROM_CHECKPOINT = "auto"`** (default) - resume when a valid checkpoint is
  mounted, cold scan when it isn't. Nothing to flip between runs.
- **`True`** - require a checkpoint; raise rather than silently cold scan.
- **`False`** - always cold scan, ignoring any attached checkpoint.

A checkpoint is only used when its `gold_ids.json` matches the sample this run just drew.
The checkpointed gold docs are keyed to the sample that produced them, so a checkpoint
built at a different `N_QUESTIONS` is discarded rather than mixed in.

**Why the safety valve exists.** The N=500 run scanned 5,893,807 / 5,903,530 docs (99.84%)
over 3.4 hours, found 487/488 gold docs and filled the 5000-doc reservoir, then died on a
transient `ClientResponseError` from the HuggingFace streaming endpoint ~9,700 docs from
the end. The `finally:` block had checkpointed everything, but the exception propagated, so
`gold_doc_records, distractor_docs = ...` never bound and every downstream cell was skipped.
The retry then *also* cold-scanned, because the kernel was pushed ~60s after
`kaggle datasets create` while Kaggle still reported the dataset as "being created" -- the
mount simply wasn't there yet. The mount path itself was correct, confirmed by walking
`/kaggle/input` from a throwaway kernel. Poll
`kaggle datasets status mehedikhan72/rag-faithfulness-scan-checkpoint` until it prints
`ready` before pushing anything that depends on it.

**Why resuming mid-scan is not cheap.** `IterableDataset.skip(n)` is lazy - it still pulls
every skipped record over HTTP. A resume at 99.8% would re-download almost the whole dump.
Stage 3 therefore treats a scan that reached >=99% with a full reservoir as complete: any
gold id still outstanding at that point is absent from this dump version rather than merely
unreached, which is why the scan never hit its early-break in the first place.

In [ ]:
import os
import shutil

# "auto" -> resume if a valid checkpoint is found, otherwise cold scan.
# True    -> require a checkpoint; fail rather than silently cold-scan.
# False   -> always cold scan, ignoring any attached checkpoint.
RESUME_FROM_CHECKPOINT = "auto"

CHECKPOINT_FILES = [
    "gold_docs_checkpoint.json",
    "distractor_reservoir_checkpoint.json",
    "gold_ids.json",
    "sampled_questions.json",
    "scan_progress.json",
]
COLD_SCAN_NOTE = "cold scan of KILT Wikipedia (~3.4 hours at ~480 docs/sec)"


def find_checkpoint_dir(root="/kaggle/input"):
    """Locate the checkpoint by its CONTENTS, not by a hardcoded mount path.

    Hardcoding `/kaggle/input/datasets/<owner>/<slug>` proved unreliable: a throwaway
    kernel found the dataset there while this notebook, pushed minutes later with an
    identical `dataset_sources`, saw nothing at that path. Rather than keep guessing at
    Kaggle's mount layout, walk the input tree and match on the marker files. Costs
    milliseconds and is immune to whatever Kaggle does next.
    """
    root = Path(root)
    if not root.exists():
        return None
    for dirpath, dirnames, filenames in os.walk(root):
        names = set(filenames)
        if {"scan_progress.json", "gold_docs_checkpoint.json"} <= names:
            found = Path(dirpath)
            missing = [n for n in CHECKPOINT_FILES if n not in names]
            if missing:
                print(f"  found partial checkpoint at {found}, missing {missing} - ignoring")
                continue
            return found
    return None


def show_input_tree(root="/kaggle/input", max_depth=4):
    root = Path(root)
    if not root.exists():
        print(f"  {root} does not exist at all")
        return
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.count("/") - str(root).count("/")
        if depth > max_depth:
            dirnames[:] = []
            continue
        print("  " * (depth + 1) + dirpath)
        for f in sorted(filenames)[:6]:
            print("  " * (depth + 2) + f"- {f}")


CHECKPOINT_DIR = None if RESUME_FROM_CHECKPOINT is False else find_checkpoint_dir()

if RESUME_FROM_CHECKPOINT is True and CHECKPOINT_DIR is None:
    print("No checkpoint found. /kaggle/input contains:")
    show_input_tree()
    raise RuntimeError(
        f"RESUME_FROM_CHECKPOINT is True but no checkpoint was found under /kaggle/input.\n"
        f"Refusing to fall through to a {COLD_SCAN_NOTE}.\n"
        f"Confirm the dataset is attached and `kaggle datasets status "
        f"mehedikhan72/rag-faithfulness-scan-checkpoint` prints `ready` before pushing, "
        f"or set RESUME_FROM_CHECKPOINT = False."
    )

resumed = False

if CHECKPOINT_DIR is not None:
    print(f"Checkpoint found at {CHECKPOINT_DIR}")
    for name in CHECKPOINT_FILES:
        shutil.copy(CHECKPOINT_DIR / name, OUT_DIR / name)
        print(f"  restored {name} ({(CHECKPOINT_DIR / name).stat().st_size / 1e6:.2f} MB)")

    # Stage 2 re-sampled and rebound `questions` / `gold_ids` a moment ago. The
    # checkpointed gold docs are keyed to the sample that produced them, so the two must
    # agree before the checkpoint can be trusted -- one built at a different N_QUESTIONS
    # would otherwise be silently mixed into this run.
    with open(OUT_DIR / "gold_ids.json") as f:
        checkpoint_gold_ids = set(json.load(f))

    if checkpoint_gold_ids == set(gold_ids):
        with open(OUT_DIR / "sampled_questions.json") as f:
            questions = json.load(f)
        gold_ids = checkpoint_gold_ids
        state = json.load(open(OUT_DIR / "scan_progress.json"))
        n_gold = len(json.load(open(OUT_DIR / "gold_docs_checkpoint.json")))
        n_dist = len(json.load(open(OUT_DIR / "distractor_reservoir_checkpoint.json")))
        print(f"\nCheckpoint matches this sample - resuming.")
        print(f"  scanned so far : {state['scanned']:,}")
        print(f"  questions      : {len(questions)} | gold ids needed: {len(gold_ids)}")
        print(f"  gold on hand   : {n_gold} | distractors on hand: {n_dist}")
        resumed = True
    else:
        # Wrong checkpoint for this configuration. Remove the restored scan state, or
        # Stage 3 would resume against a sample that isn't this one.
        for name in CHECKPOINT_FILES:
            if name not in ("sampled_questions.json", "gold_ids.json"):
                (OUT_DIR / name).unlink(missing_ok=True)
        with open(OUT_DIR / "sampled_questions.json", "w") as f:
            json.dump(questions, f, indent=2)
        with open(OUT_DIR / "gold_ids.json", "w") as f:
            json.dump(sorted(gold_ids), f)
        print(f"\nWARNING: checkpoint holds {len(checkpoint_gold_ids)} gold ids but this "
              f"run sampled {len(set(gold_ids))}.")
        print(f"Checkpoint discarded (it belongs to a different sample).")

if not resumed:
    reason = ("RESUME_FROM_CHECKPOINT = False" if RESUME_FROM_CHECKPOINT is False
              else "no checkpoint found under /kaggle/input" if CHECKPOINT_DIR is None
              else "checkpoint did not match this sample")
    print(f"\nCold run ({reason}): {COLD_SCAN_NOTE}.")
    if CHECKPOINT_DIR is None and RESUME_FROM_CHECKPOINT is not False:
        print("/kaggle/input contains:")
        show_input_tree()

## Stage 3 — Shrink KILT Wikipedia: stream once, keep gold docs + reservoir-sampled distractors

**Fixed from the original draft:** `text` is a dict `{"paragraph": [...]}`, not a flat list -- the join logic below now matches the real schema (confirmed against the HF dataset card).

In [ ]:
import time

GOLD_CHECKPOINT = OUT_DIR / "gold_docs_checkpoint.json"
DISTRACTOR_CHECKPOINT = OUT_DIR / "distractor_reservoir_checkpoint.json"
SCAN_PROGRESS = OUT_DIR / "scan_progress.json"
KILT_WIKI_TOTAL_DOCS = 5_903_530

def collect_gold_and_distractors(gold_ids, n_distractors=N_DISTRACTORS, progress_every=10000):
    gold_ids = set(gold_ids)

    found = {}
    reservoir = []
    seen_non_gold = 0
    start_at = 0

    if GOLD_CHECKPOINT.exists():
        with open(GOLD_CHECKPOINT) as f:
            for d in json.load(f):
                found[d["wikipedia_id"]] = d
    if DISTRACTOR_CHECKPOINT.exists():
        with open(DISTRACTOR_CHECKPOINT) as f:
            reservoir = json.load(f)
    if SCAN_PROGRESS.exists():
        state = json.load(open(SCAN_PROGRESS))
        start_at = state["scanned"]
        seen_non_gold = state["seen_non_gold"]

    remaining = gold_ids - set(found.keys())

    # The scan can be complete-enough to skip entirely. That happens when the reservoir is
    # full and we have already walked essentially the whole dump -- any gold id still
    # outstanding at that point is absent from this dump version, not merely unreached, so
    # more streaming cannot find it. Returning here matters because resuming is not cheap:
    # IterableDataset.skip(n) is lazy and re-downloads every skipped record over HTTP, so a
    # "resume" at 99.8% would cost nearly as much as the original 3.4-hour scan.
    if start_at >= KILT_WIKI_TOTAL_DOCS * 0.99 and len(reservoir) >= n_distractors:
        print(f"Scan already complete: {start_at:,}/{KILT_WIKI_TOTAL_DOCS:,} docs "
              f"({100 * start_at / KILT_WIKI_TOTAL_DOCS:.2f}%), "
              f"{len(found)}/{len(gold_ids)} gold, {len(reservoir)} distractors.")
        if remaining:
            print(f"  {len(remaining)} gold id(s) never found (dump version drift, not a "
                  f"bug): {sorted(remaining)}")
        json.dump(sorted(remaining), open(OUT_DIR / "missing_gold_ids.json", "w"))
        return list(found.values()), reservoir

    if start_at:
        print(f"Resuming: {len(found)}/{len(gold_ids)} gold found, "
              f"{len(reservoir)} distractors held, skipping ahead {start_at} records.")

    ds = load_dataset("facebook/kilt_wikipedia", split="full", streaming=True, trust_remote_code=True)
    if start_at:
        ds = ds.skip(start_at)

    def extract_text(doc):
        return " ".join(doc.get("text", {}).get("paragraph", []))

    scanned = start_at
    t0 = time.time()
    def checkpoint_gold():
        with open(GOLD_CHECKPOINT, "w") as f:
            json.dump(list(found.values()), f)

    def checkpoint_reservoir():
        with open(DISTRACTOR_CHECKPOINT, "w") as f:
            json.dump(reservoir, f)

    try:
        for doc in ds:
            scanned += 1
            doc_id = doc["wikipedia_id"]

            if doc_id in remaining:
                record = {"wikipedia_id": doc_id, "title": doc.get("wikipedia_title", ""),
                          "text": extract_text(doc)}
                found[doc_id] = record
                checkpoint_gold()
                remaining.discard(doc_id)
                print(f"  [gold {len(found)}/{len(gold_ids)}] wikipedia_id={doc_id} at scan {scanned}")
            elif doc_id not in gold_ids:
                seen_non_gold += 1
                record = {"wikipedia_id": doc_id, "title": doc.get("wikipedia_title", ""),
                          "text": extract_text(doc)}
                if len(reservoir) < n_distractors:
                    reservoir.append(record)
                else:
                    j = random.randint(0, seen_non_gold - 1)   # true reservoir sampling
                    if j < n_distractors:
                        reservoir[j] = record

            if scanned % progress_every == 0:
                elapsed = time.time() - t0
                rate = (scanned - start_at) / elapsed if elapsed > 0 else 0
                pct = 100 * scanned / KILT_WIKI_TOTAL_DOCS
                eta_min = (KILT_WIKI_TOTAL_DOCS - scanned) / rate / 60 if rate > 0 else float("inf")
                json.dump({"scanned": scanned, "seen_non_gold": seen_non_gold}, open(SCAN_PROGRESS, "w"))
                checkpoint_reservoir()
                print(f"  scanned {scanned:>9,}/{KILT_WIKI_TOTAL_DOCS:,} ({pct:5.1f}%) | "
                      f"{rate:6.0f} docs/sec | gold {len(found)}/{len(gold_ids)} | "
                      f"reservoir {len(reservoir)}/{n_distractors} | ETA: {eta_min:6.1f} min")

            if not remaining and len(reservoir) >= n_distractors and scanned - start_at > n_distractors * 5:
                # gold done AND reservoir full AND we've mixed it enough post-fill for genuine randomization
                print(f"Done: all gold found, reservoir full, after {scanned} records.")
                break
    finally:
        checkpoint_gold()
        json.dump({"scanned": scanned, "seen_non_gold": seen_non_gold}, open(SCAN_PROGRESS, "w"))
        checkpoint_reservoir()

    still_missing = gold_ids - set(found.keys())
    json.dump(sorted(still_missing), open(OUT_DIR / "missing_gold_ids.json", "w"))
    if still_missing:
        print(f"NOTE: {len(still_missing)} gold ids never found (dump version drift, not a bug). "
              f"Drop their questions in Stage 3b.")

    return list(found.values()), reservoir

gold_doc_records, distractor_docs = collect_gold_and_distractors(gold_ids)

In [ ]:
def merge_into_corpus():
    with open(GOLD_CHECKPOINT) as f:
        gold_records = json.load(f)

    with open(DISTRACTOR_CHECKPOINT) as f:
        distractor_records = json.load(f)

    all_docs = gold_records + distractor_records
    random.shuffle(all_docs)

    # Corpus and chunks are machine-only bulk data (~17MB / ~38MB), so they are written
    # unindented. Everything a human actually reads gets indent=2.
    with open(OUT_DIR / "corpus.json", "w") as f:
        json.dump([
            {
                "wikipedia_id": doc["wikipedia_id"],
                "title": doc.get("title", ""),
                "text": doc.get("text", ""),
            }
            for doc in all_docs
        ], f)

    print(f"Merged corpus: {len(gold_records)} gold docs + {len(distractor_records)} distractors "
          f"= {len(all_docs)} total documents -> corpus.json")

merge_into_corpus()

In [ ]:
with open(OUT_DIR / "missing_gold_ids.json") as f:
    missing_ids = set(json.load(f))

if missing_ids:
    with open(OUT_DIR / "sampled_questions.json") as f:
        all_questions = json.load(f)

    kept = [q for q in all_questions if not (set(q["gold_wikipedia_ids"]) & missing_ids)]
    dropped = len(all_questions) - len(kept)

    with open(OUT_DIR / "sampled_questions.json", "w") as f:
        json.dump(kept, f, indent=2)

    print(f"Dropped {dropped} unanswerable questions, kept {len(kept)}.")
else:
    print("No missing gold ids -- nothing to drop.")

## Stage 4 — Chunk into ~100-token passages (title prepended for display, per Wallat et al.)

Each chunk carries two text fields: `text` (`"{title}: {piece}"`, shown to the model at
generation time — matches Wallat et al.'s convention of prepending the title for
disambiguation) and `body_text` (just `piece`, no title). Stage 5 indexes on `body_text`
only — see that cell's note for why.

In [ ]:
def chunk_corpus(chunk_size=CHUNK_SIZE_TOKENS):
    chunks = []
    with open(OUT_DIR / "corpus.json") as f:
        corpus = json.load(f)
    for doc in corpus:
        words = doc["text"].split()
        for i in range(0, len(words), chunk_size):
            piece = " ".join(words[i:i + chunk_size])
            if not piece.strip():
                continue
            chunks.append({
                "chunk_id": f"{doc['wikipedia_id']}_{i // chunk_size}",
                "wikipedia_id": doc["wikipedia_id"],
                "title": doc["title"],
                "text": f"{doc['title']}: {piece}",
                "body_text": piece,
            })

    with open(OUT_DIR / "chunks.json", "w") as f:
        json.dump(chunks, f)

    print(f"Chunked corpus into {len(chunks)} passages of ~{chunk_size} tokens.")
    return chunks

chunks = chunk_corpus()

## Stage 5 — Build the BM25 index

**Indexes `body_text`, not `text` (added 2026-08-08).** Indexing on `text` (title+body)
meant every chunk of a document shared the same title terms, so BM25 gave every chunk of a
matching document a similar score boost regardless of whether that specific 100-word piece
was actually relevant — confirmed by Stage 6b: 52% of questions had all 5 top-k slots filled
by chunks of a single document, which starves the Step 5 adversarial test of a second
distinct retrieved document to work with, and can bury the one chunk that actually states
the answer under other same-document chunks that only "match" via the shared title.
Indexing on body text alone lets chunks compete on their own content.

In [ ]:
_TOKEN_RE = re.compile(r"\w+")

def _tokenize(text):
    return _TOKEN_RE.findall(text.lower())

def build_bm25_index():
    with open(OUT_DIR / "chunks.json") as f:
        all_chunks = json.load(f)

    tokenized = [_tokenize(c["body_text"]) for c in all_chunks]
    bm25 = BM25Okapi(tokenized)
    print(f"BM25 index built over {len(all_chunks)} chunks (indexed on body_text).")
    return bm25, all_chunks

bm25, indexed_chunks = build_bm25_index()

## Stage 6 — Retrieve top-k passages for a question + sanity check

In [ ]:
def retrieve_top_k(bm25, chunks, query, k=5):
    scores = bm25.get_scores(_tokenize(query))
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [chunks[i] for i in top_indices]

with open(OUT_DIR / "sampled_questions.json") as f:
    first_q = json.load(f)[0]

top5 = retrieve_top_k(bm25, indexed_chunks, first_q["question"], k=5)

print("--- Sanity check ---")
print("Question:", first_q["question"])
print("Gold wikipedia ids:", first_q["gold_wikipedia_ids"])
print("Top-5 retrieved wikipedia ids:", [c["wikipedia_id"] for c in top5])

## Stage 6b — Diagnostic: document diversity in top-5 retrieval

BM25 chunks are prefixed with their document title (Stage 4), so every chunk of the true
matching article gets a title-term boost — top-5 can end up as several chunks of the *same*
document rather than 5 distinct documents. This matters for the Step 5 adversarial test
(context.md §4), which needs a *retrieved-but-never-cited different document* per question —
that variant isn't constructible for questions where top-5 is single-document. Run across
**all** sampled questions (not just one spot-check) and persist the result for the
methodology write-up.

In [ ]:
from collections import Counter

with open(OUT_DIR / "sampled_questions.json") as f:
    all_questions = json.load(f)

unique_doc_counts = []
gold_hit_at_5 = 0
single_doc_dominates = []  # questions where all 5 top chunks are from ONE document

for q in all_questions:
    top5 = retrieve_top_k(bm25, indexed_chunks, q["question"], k=5)
    doc_ids = [c["wikipedia_id"] for c in top5]
    n_unique = len(set(doc_ids))
    unique_doc_counts.append(n_unique)

    if set(q["gold_wikipedia_ids"]) & set(doc_ids):
        gold_hit_at_5 += 1

    if n_unique == 1:
        single_doc_dominates.append({
            "id": q["id"],
            "question": q["question"],
            "doc_id": doc_ids[0],
            "gold_ids": q["gold_wikipedia_ids"],
            "is_gold": doc_ids[0] in q["gold_wikipedia_ids"],
        })

n = len(all_questions)
dist = Counter(unique_doc_counts)
n_single_and_gold = sum(1 for x in single_doc_dominates if x["is_gold"])

report = {
    "n_questions": n,
    "unique_doc_count_distribution": {str(k): v for k, v in sorted(dist.items())},
    "mean_unique_docs_in_top5": sum(unique_doc_counts) / n,
    "gold_hit_at_5": gold_hit_at_5,
    "gold_hit_at_5_pct": 100 * gold_hit_at_5 / n,
    "n_single_document_top5": len(single_doc_dominates),
    "n_single_document_top5_pct": 100 * len(single_doc_dominates) / n,
    "n_single_document_top5_where_gold": n_single_and_gold,
    "single_document_cases": single_doc_dominates,
}

with open(OUT_DIR / "retrieval_diversity_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("=== Unique documents among top-5 chunks, distribution across all questions ===")
for k in sorted(dist):
    print(f"  {k} unique doc(s) in top-5: {dist[k]:3d} questions ({100*dist[k]/n:.1f}%)")

print(f"\nMean unique docs in top-5: {sum(unique_doc_counts)/n:.2f} / 5")
print(f"Gold doc present in top-5: {gold_hit_at_5}/{n} ({100*gold_hit_at_5/n:.1f}%)")
print(f"\nQuestions where ALL 5 top chunks are from the SAME single document: "
      f"{len(single_doc_dominates)}/{n} ({100*len(single_doc_dominates)/n:.1f}%)")
print(f"  ...of those, the single document IS the gold doc: "
      f"{n_single_and_gold}/{len(single_doc_dominates)}")
print(f"\nSaved full report -> {OUT_DIR / 'retrieval_diversity_report.json'}")

## Stage 7 — Persist outputs

`/kaggle/working` is preserved automatically when you commit/save a version, so no extra action is required -- but if you want everything zipped for easy download or upload as a Kaggle Dataset for the next notebook (generation stage), run this:

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/retrieval_pipeline_data", "zip", OUT_DIR)
print("Zipped to /kaggle/working/retrieval_pipeline_data.zip")